In [1]:
import pandas as pd
import os


In [2]:
BASE_CODE_FOLDER=os.sep.join(os.getcwd().split(os.sep)[:os.getcwd().split(os.sep).index('Mutual_Fund_Analysis')+1])
DATASET_FOLDER=os.path.join(BASE_CODE_FOLDER, 'Dataset')
PORTFOLIO_DISTRIBUTION=os.path.join(DATASET_FOLDER,'Monthly Reports October 2024')
PORTFOLIO_DISTRIBUTION

'/media/vchopra/DATA/Complete Technical Work/All  Projects Implemented/Major Projects/Mutual_Fund_Analysis/Dataset/Monthly Reports October 2024'

In [3]:
os.listdir(PORTFOLIO_DISTRIBUTION)

['360_one_asset.xlsx',
 'Adityabirla.xls',
 'Axis.xlsx',
 'bank of india.xlsx',
 'icici',
 'motilal_oswal.xls',
 'NIMF-MONTHLY-PORTFOLIO-31-Oct-24.xls',
 'ppfas.xlsx',
 'quant.xlsx']

In [4]:
CURRENT_FILE=os.path.join(PORTFOLIO_DISTRIBUTION,'ppfas.xlsx')

In [23]:
def preprocess_excel_data(df):
    """
    Cleans and preprocesses an Excel sheet DataFrame for extracting stock names.
    
    Args:
        df (pd.DataFrame): The raw DataFrame loaded from an Excel sheet.

    Returns:
        str: Preprocessed and cleaned data as a string, ready for LLM input.
    """
    # Step 1: Drop rows with missing values in the "Name of the Instrument" column
    if "Name of the Instrument" in df.columns:
        df = df.dropna(subset=["Name of the Instrument"])
    else:
        raise ValueError("The column 'Name of the Instrument' is missing from the DataFrame.")

    # Step 2: Keep only relevant columns
    relevant_columns = ["Name of the Instrument"]
    df = df[relevant_columns]

    # Step 3: Remove duplicates
    df = df.drop_duplicates()

    # Step 4: Standardize text (strip whitespace, remove special characters if necessary)
    df["Name of the Instrument"] = df["Name of the Instrument"].str.strip()

    # Step 5: Handle header issues (if the header row appears in data, drop it)
    if "Name of the Instrument" in df["Name of the Instrument"].values:
        df = df[df["Name of the Instrument"] != "Name of the Instrument"]

    # Convert the cleaned DataFrame to a string for LLM input
    cleaned_data = df.to_string(index=False, header=False)
    return cleaned_data


In [43]:
def read_excel_and_save_sheets(file_path):
    # Load the Excel file
    excel_data = pd.ExcelFile(file_path)
    
    # # Create the output folder if it doesn't exist
    # if not os.path.exists(output_folder):
    #     os.makedirs(output_folder)
    excel_data_dict={}
    # Iterate through each sheet in the Excel file
    for sheet_name in excel_data.sheet_names:
        # Load the sheet into a DataFrame
        sheet_df = excel_data.parse(sheet_name)
        # print(type(sheet_df), sheet_df.columns)
        if 'Unnamed: 0' in list(sheet_df.columns):
            del sheet_df['Unnamed: 0']
        # Convert the DataFrame to a string
        sheet_df.fillna("", inplace=True)
        sheet_data_string = sheet_df.to_string(index=False)
        print(sheet_df.head().to_string(index=False))
        sheet_data_string.replace("NaN", "")
        excel_data_dict[sheet_name]=sheet_data_string
        # Save the string to a text file
        # output_file = os.path.join(output_folder, f"{sheet_name}.txt")
        # with open(output_file, "w") as file:
        #     file.write(sheet_data_string)
        
        print(f"Current Sheet '{sheet_name}' processed to string")
        print()
    return excel_data_dict
excel_data=read_excel_and_save_sheets(CURRENT_FILE)

 Sr No. Short Name                           Scheme Name
      1      PPFCF           Parag Parikh Flexi Cap Fund
      2       PPLF              Parag Parikh Liquid Fund
      3      PPTSF      Parag Parikh ELSS Tax Saver Fund
      4      PPCHF Parag Parikh Conservative Hybrid Fund
      5       PPAF           Parag Parikh Arbitrage Fund
Current Sheet 'Index' processed to string

Parag Parikh Flexi Cap Fund (An open-ended dynamic equity scheme investing across large cap, mid-cap, small-cap stocks) Unnamed: 2        Unnamed: 3 Unnamed: 4                         Unnamed: 5        Unnamed: 6 Unnamed: 7 Unnamed: 8 Unnamed: 9 Unnamed: 10
                                                                                                                                                                                                                                                                 
                                                                     Monthly Portfolio Statement as

/tmp/ipykernel_103013/2877269545.py:17: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  sheet_df.fillna("", inplace=True)


In [44]:
# excel_data = pd.ExcelFile(CURRENT_FILE)
# sheet_df_list=[]
# for sheet_name in excel_data.sheet_names:
#     # Load the sheet into a DataFrame
#     sheet_df = excel_data.parse(sheet_name)
#     # print(type(sheet_df), sheet_df.columns)
#     if 'Unnamed: 0' in list(sheet_df.columns):
#         del sheet_df['Unnamed: 0']
#     # Convert the DataFrame to a string
#     sheet_df_list.append(sheet_df)
#     # sheet_data_string = sheet_df.to_string(index=False)
#     print(sheet_df.head().to_string(index=False))

In [45]:
# sheet_df_list[1]

In [46]:
excel_data.keys()

dict_keys(['Index', 'PPFCF', 'PPLF', 'PPTSF', 'PPCHF', 'PPAF', 'PPDAAF'])

In [47]:
# Convert DataFrame to a string format
# data_text = df.to_string(index=False)

### Cleaning Data

In [48]:
excel_data['PPFCF']

"                                                                                                                                                           Parag Parikh Flexi Cap Fund (An open-ended dynamic equity scheme investing across large cap, mid-cap, small-cap stocks)                     Unnamed: 2                                        Unnamed: 3                                     Unnamed: 4                                          Unnamed: 5                                   Unnamed: 6                          Unnamed: 7              Unnamed: 8      Unnamed: 9     Unnamed: 10\n                                                                                                                                                                                                                                                                                                                                                                                                                      

In [75]:
import osfrom huggingface_hub import loginhf_token = os.getenv("HF_TOKEN")if hf_token:    login(token=hf_token)else:    print("Set HF_TOKEN to authenticate with Hugging Face.")

### Setting up Tranformers

In [77]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM,AutoModelForCausalLM

# Load the model and tokenizer
model_name = "meta-llama/Llama-3.2-1B-Instruct"  # You can also use "t5-small", "facebook/bart-large", etc.
tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

KeyboardInterrupt: 

In [ ]:
def prepare_prompt(excel_text):
    prompt = f"""
    Here is the breakup distribution of Parag Parikh Flexi Cap Fund.  
    What are the types of insturment the fund has invested in ?
    {excel_text}
    Return the stock names as a comma-separated list.
    """
    return prompt

In [ ]:
def extract_stocks_with_hf_model(excel_text):
    # Prepare the prompt
    prompt = prepare_prompt(excel_text)
    
    # Tokenize the input
    inputs = tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True)
    
    # Generate the output
    outputs = model.generate(inputs.input_ids, max_length=512, num_beams=5, early_stopping=True)
    
    # Decode and return the output
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


In [ ]:
# Use Hugging Face text-to-text model to extract stocks
stock_list = extract_stocks_with_hf_model(excel_data['PPFCF'])
print("Extracted Stocks:", str(stock_list))

Extracted Stocks: Axis Bank Limited INE238A01034 Banks 24845558 288096.67 0.0352 Kotak Mahindra Bank Limited INE237A01028 Banks 13406651 232082.54 0.0212 Motilal Oswal Financial Services Limited INE338I01027 Capital Markets 18554307 173909.52 0.0212 Infosys Limited INE009A01021 IT - Software 9268754


In [ ]:
import os
from huggingface_hub import login

hf_token = os.getenv("HF_TOKEN")
if hf_token:
    login(token=hf_token)
else:
    print("Set HF_TOKEN to authenticate with Hugging Face.")

### Setting up model

In [68]:
import ollama

# Initialize the Ollama client
client = ollama.Client()


In [69]:
def extract_information(prompt, data):
    # Combine the prompt with the data
    input_text = f"{prompt}\n\n{data}"
    # Generate a response using the Llama model
    response = client.generate(model='llama3.2', prompt=input_text)
    return response['response']

In [70]:
prompt = "Extract the list of stocks present in the Parag Parikh Flexi Cap Fund"

In [71]:
result = extract_information(prompt, excel_data['PPFCF'])
print(result)

KeyboardInterrupt: 

In [ ]:
def process_unstructured_data(data, prompt):
    response = client.generate(model="llama3.2", prompt=f"{prompt}\n\n{data}")
    return response['response']

# Example unstructured data
data = """
Parag Parikh Flexi Cap Fund contains stocks such as HDFC Bank, Infosys Limited, 
and Maruti Suzuki. It also includes lesser-known stocks like Indian Energy Exchange.
"""

prompt = "Extract all stock names from the following text and return as a list."


In [ ]:
def extract_stocks_from_excel(excel_text):
    # Pre-process Excel file
    # excel_text = extract_excel_data(file_path, sheet_name)

    # Define the prompt for Llama
    prompt = """
    Extract the list of stocks from the following portfolio data. 
    Provide the output as a comma-separated list of stock names.
    """

    # Query the Llama model via Ollama
    stocks_output = query_llama(prompt, excel_text)
    return stocks_output


stocks_list = extract_stocks_from_excel(excel_data['PPFCF'])
print("Extracted Stocks:", stocks_list)
